# Reliability Testing

This notebook identifies severe winter dunkelflaute seasons from the local ERA5 renewable-capacity-factor data, selects a representative winter year, and compares how the `with_h2` and `no_h2` scenarios respond over the winter months.

Focus:
- winter renewable capacity-factor stress
- daily generation by technology
- daily storage-level fluctuation in each scenario

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import src.assumptions as A
import src.matplotlib_style  # noqa: F401
from src.data import renewable_capacity_factors
from src.demand_model import DemandMode, predicted_demand
from src.power_system import PowerSystem
from src.power_system_no_h2 import PowerSystemNoH2
from src.units import Units as U


In [ ]:
# Scenario configuration
REN_GW = 200
DAC_GW = 2
DEMAND_MODE = DemandMode.SEASONAL
CAPACITY_FACTORS_SOURCE = "era5_2024"
ENABLE_IMPORTS = False

HYDROGEN_STORAGE_TWH = A.HydrogenStorage.CavernStorage.Capacity
ELECTROLYSER_GW = A.HydrogenStorage.Electrolysis.Power
HYDROGEN_GEN_GW = A.HydrogenStorage.Generation.Power

WINTER_MONTHS = (11, 12, 1, 2)
JAN_FEB_MONTHS = (1, 2)
DUNKELFLAUTE_THRESHOLD = 0.15
TOP_N_WORST_WINTERS = 12
SELECTED_WINTER_YEAR = 2010  # Set to None to auto-select a representative winter.

OUTPUT_DIR = Path("nbs/output/reliability_testing")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def as_float_series(series: pd.Series) -> pd.Series:
    if hasattr(series, "pint"):
        return pd.Series(series.pint.magnitude, index=series.index, name=series.name)
    return pd.Series(series.astype(float), index=series.index, name=series.name)


def winter_year_index(index: pd.DatetimeIndex) -> pd.Index:
    winter_year = index.year.astype(int)
    winter_year = np.where(index.month.isin(JAN_FEB_MONTHS), winter_year - 1, winter_year)
    return pd.Index(winter_year, name="winter_year")


def combined_capacity_factor(cf: pd.DataFrame) -> pd.Series:
    weights = {
        "solar": float(A.Renewables.CapacityRatios.Solar),
        "onshore": float(A.Renewables.CapacityRatios.OnshoreWind),
        "offshore": float(A.Renewables.CapacityRatios.OffshoreWind),
    }
    total_weight = sum(weights.values())
    return sum(cf[col] * weight for col, weight in weights.items()) / total_weight


def build_generation_breakdown(capacity_factors: pd.DataFrame, renewable_capacity_gw: int) -> pd.DataFrame:
    renewable_capacity = renewable_capacity_gw * U.GW

    solar = (
        renewable_capacity
        * A.Renewables.CapacityRatios.Solar
        * capacity_factors["solar"]
        * A.HoursPerDay
    ).astype("pint[TWh]")
    onshore = (
        renewable_capacity
        * A.Renewables.CapacityRatios.OnshoreWind
        * capacity_factors["onshore"]
        * A.HoursPerDay
    ).astype("pint[TWh]")
    offshore = (
        renewable_capacity
        * A.Renewables.CapacityRatios.OffshoreWind
        * capacity_factors["offshore"]
        * A.HoursPerDay
    ).astype("pint[TWh]")

    nuclear_daily = (A.Nuclear.Capacity * A.Nuclear.CapacityFactor * A.HoursPerDay).to(U.TWh)
    nuclear = pd.Series(
        np.full(len(capacity_factors.index), nuclear_daily.magnitude),
        index=capacity_factors.index,
        dtype="pint[TWh]",
        name="nuclear",
    )

    breakdown = pd.DataFrame(
        {
            "solar_generation": solar,
            "onshore_generation": onshore,
            "offshore_generation": offshore,
            "nuclear_generation": nuclear,
        },
        index=capacity_factors.index,
    )
    breakdown["total_generation"] = breakdown.sum(axis=1)
    return breakdown


def simulation_series(sim_df: pd.DataFrame, renewable_capacity_gw: int, stem: str, unit: str = "TWh") -> pd.Series:
    return as_float_series(sim_df[f"{stem} ({unit}),RC={renewable_capacity_gw}GW"])


def storage_level_series(sim_df: pd.DataFrame, renewable_capacity_gw: int, stem: str) -> pd.Series:
    return as_float_series(sim_df[f"{stem} (TWh),RC={renewable_capacity_gw}GW"])


def build_winter_summary(label: str, sim_df: pd.DataFrame, renewable_capacity_gw: int) -> dict[str, float]:
    summary = {
        "scenario": label,
        "min_medium_storage_TWh": storage_level_series(
            sim_df, renewable_capacity_gw, "medium_storage_level"
        ).min(),
        "total_gas_ccs_TWh": simulation_series(
            sim_df, renewable_capacity_gw, "gas_ccs_energy"
        ).sum(),
        "total_dac_energy_TWh": simulation_series(
            sim_df, renewable_capacity_gw, "dac_energy"
        ).sum(),
        "total_curtailed_TWh": simulation_series(
            sim_df, renewable_capacity_gw, "curtailed_energy"
        ).sum(),
    }

    h2_col = f"hydrogen_storage_level (TWh),RC={renewable_capacity_gw}GW"
    if h2_col in sim_df.columns:
        summary["min_hydrogen_storage_TWh"] = as_float_series(sim_df[h2_col]).min()
    else:
        summary["min_hydrogen_storage_TWh"] = np.nan

    return summary


In [ ]:
# Load local capacity-factor history and rank winter dunkelflaute seasons.
daily_cf = renewable_capacity_factors.get_renewable_capacity_factors(
    source=CAPACITY_FACTORS_SOURCE,
    resample="D",
)
cf_float = daily_cf.astype(float)
cf_float["combined"] = combined_capacity_factor(cf_float)

winter_cf = cf_float[cf_float.index.month.isin(WINTER_MONTHS)].copy()
winter_cf["winter_year"] = winter_year_index(winter_cf.index)

counts = winter_cf.groupby("winter_year").size()
complete_winter_years = counts[counts >= 120].index
winter_cf = winter_cf[winter_cf["winter_year"].isin(complete_winter_years)]

winter_ranking = winter_cf.groupby("winter_year").agg(
    winter_mean_cf=("combined", "mean"),
    winter_min_cf=("combined", "min"),
    p10_cf=("combined", lambda s: s.quantile(0.10)),
    days_below_threshold=("combined", lambda s: int((s < DUNKELFLAUTE_THRESHOLD).sum())),
)
winter_ranking["severity_score"] = (
    winter_ranking["winter_mean_cf"].rank(method="min", ascending=True)
    + winter_ranking["winter_min_cf"].rank(method="min", ascending=True)
    + winter_ranking["p10_cf"].rank(method="min", ascending=True)
    + winter_ranking["days_below_threshold"].rank(method="min", ascending=False)
)
winter_ranking = winter_ranking.sort_values(
    ["severity_score", "winter_mean_cf", "winter_min_cf"]
)

top_winter_years = winter_ranking.head(TOP_N_WORST_WINTERS).copy()
if SELECTED_WINTER_YEAR is None:
    selected_position = len(top_winter_years) // 2
    selected_winter_year = int(top_winter_years.index[selected_position])
else:
    selected_winter_year = int(SELECTED_WINTER_YEAR)

if selected_winter_year not in winter_ranking.index:
    raise ValueError(
        f"Selected winter year {selected_winter_year} is not available in the ranked data."
    )

display(top_winter_years)
print(
    "Selected winter year:",
    selected_winter_year,
    "| mean CF:",
    round(float(winter_ranking.loc[selected_winter_year, "winter_mean_cf"]), 3),
    "| min CF:",
    round(float(winter_ranking.loc[selected_winter_year, "winter_min_cf"]), 3),
)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False, height_ratios=[1, 2])

axes[0].bar(
    top_winter_years.index.astype(str),
    top_winter_years["winter_mean_cf"],
    color=["#c44e52" if yr == selected_winter_year else "#4c72b0" for yr in top_winter_years.index],
)
axes[0].set_title("Lowest winter combined capacity-factor seasons")
axes[0].set_ylabel("Mean combined CF")
axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", rotation=45)

selected_cf = winter_cf[winter_cf["winter_year"] == selected_winter_year].copy()
axes[1].plot(selected_cf.index, selected_cf["solar"], label="Solar CF", linewidth=1.1)
axes[1].plot(selected_cf.index, selected_cf["onshore"], label="Onshore wind CF", linewidth=1.1)
axes[1].plot(selected_cf.index, selected_cf["offshore"], label="Offshore wind CF", linewidth=1.1)
axes[1].plot(selected_cf.index, selected_cf["combined"], label="Combined CF", color="black", linewidth=1.4)
axes[1].axhline(
    DUNKELFLAUTE_THRESHOLD,
    color="#c44e52",
    linestyle="--",
    linewidth=1.0,
    label=f"Dunkelflaute threshold = {DUNKELFLAUTE_THRESHOLD:.2f}",
)
below_threshold = selected_cf["combined"] < DUNKELFLAUTE_THRESHOLD
for ts, is_low in below_threshold.items():
    if is_low:
        axes[1].axvspan(ts, ts + pd.Timedelta(days=1), color="#c44e52", alpha=0.08)

axes[1].set_title(f"Capacity-factor profile for winter {selected_winter_year}-{selected_winter_year + 1}")
axes[1].set_ylabel("Capacity factor")
axes[1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axes[1].grid(alpha=0.3)
axes[1].legend(loc="upper right", ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
demand_df = predicted_demand(
    mode=DEMAND_MODE,
    historical="era5",
    filter_ldz=True,
)

generation_breakdown = build_generation_breakdown(daily_cf, REN_GW)

common_idx = generation_breakdown.index.intersection(demand_df.index)
generation_breakdown = generation_breakdown.reindex(common_idx)
demand_df = demand_df.reindex(common_idx)

winter_dates = winter_cf[winter_cf["winter_year"] == selected_winter_year].index
winter_dates = winter_dates.intersection(common_idx)

generation_winter = generation_breakdown.loc[winter_dates].copy()
demand_winter = demand_df.loc[winter_dates].copy()
cf_winter = cf_float.loc[winter_dates].copy()

net_supply_winter = (
    generation_winter["total_generation"] - demand_winter["demand"]
).to_frame(name=f"S-D(TWh),Ren={REN_GW}GW")
net_supply_input = net_supply_winter.reset_index().rename(
    columns={net_supply_winter.reset_index().columns[0]: "index"}
)

generation_winter.head()


In [ ]:
ps_h2 = PowerSystem(
    renewable_capacity=REN_GW * U.GW,
    hydrogen_storage_capacity=HYDROGEN_STORAGE_TWH,
    electrolyser_power=ELECTROLYSER_GW,
    dac_capacity=DAC_GW * U.GW,
    hydrogen_generation_power=HYDROGEN_GEN_GW,
    enable_imports=ENABLE_IMPORTS,
    capacity_factors_source=CAPACITY_FACTORS_SOURCE,
    enable_gas_ltdac=True,
)

ps_no_h2 = PowerSystemNoH2(
    renewable_capacity=REN_GW * U.GW,
    dac_capacity=DAC_GW * U.GW,
    enable_imports=ENABLE_IMPORTS,
    capacity_factors_source=CAPACITY_FACTORS_SOURCE,
)

sim_h2 = ps_h2.run_simulation(net_supply_input)
sim_no_h2 = ps_no_h2.run_simulation(net_supply_input)

failed = []
if sim_h2 is None:
    failed.append("with_h2")
if sim_no_h2 is None:
    failed.append("no_h2")
if failed:
    raise RuntimeError(
        "Simulation failed for: "
        + ", ".join(failed)
        + ". Increase renewable capacity, storage, or enable imports."
    )

sim_h2.index = winter_dates
sim_no_h2.index = winter_dates

winter_summary = pd.DataFrame(
    [
        build_winter_summary("with_h2", sim_h2, REN_GW),
        build_winter_summary("no_h2", sim_no_h2, REN_GW),
    ]
).set_index("scenario")

winter_summary


In [ ]:
winter_summary.to_csv(OUTPUT_DIR / "winter_summary.csv")

fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharex="col")

gen_cols = [
    ("solar_generation", "Solar"),
    ("onshore_generation", "Onshore wind"),
    ("offshore_generation", "Offshore wind"),
    ("nuclear_generation", "Nuclear"),
]

for col, label in gen_cols:
    axes[0, 0].plot(
        generation_winter.index,
        as_float_series(generation_winter[col]),
        label=label,
        linewidth=1.0,
    )
axes[0, 0].plot(
    sim_h2.index,
    simulation_series(sim_h2, REN_GW, "gas_ccs_energy"),
    label="Gas CCS",
    linewidth=1.2,
    color="#c44e52",
)
axes[0, 0].plot(
    demand_winter.index,
    as_float_series(demand_winter["demand"]),
    label="Demand",
    linewidth=1.3,
    color="black",
    linestyle="--",
)
axes[0, 0].set_title("With H2: daily winter generation response")
axes[0, 0].set_ylabel("Energy per day (TWh)")
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend(loc="upper right", ncol=2)

axes[1, 0].plot(
    sim_h2.index,
    storage_level_series(sim_h2, REN_GW, "medium_storage_level"),
    label="Medium storage level",
    linewidth=1.2,
    color="#dd8452",
)
axes[1, 0].plot(
    sim_h2.index,
    storage_level_series(sim_h2, REN_GW, "hydrogen_storage_level"),
    label="Hydrogen storage level",
    linewidth=1.2,
    color="#55a868",
)
axes[1, 0].set_title("With H2: storage fluctuation")
axes[1, 0].set_ylabel("Stored energy (TWh)")
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend(loc="upper right")

for col, label in gen_cols:
    axes[0, 1].plot(
        generation_winter.index,
        as_float_series(generation_winter[col]),
        label=label,
        linewidth=1.0,
    )
axes[0, 1].plot(
    sim_no_h2.index,
    simulation_series(sim_no_h2, REN_GW, "gas_ccs_energy"),
    label="Gas CCS",
    linewidth=1.2,
    color="#c44e52",
)
axes[0, 1].plot(
    demand_winter.index,
    as_float_series(demand_winter["demand"]),
    label="Demand",
    linewidth=1.3,
    color="black",
    linestyle="--",
)
axes[0, 1].set_title("No H2: daily winter generation response")
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend(loc="upper right", ncol=2)

axes[1, 1].plot(
    sim_no_h2.index,
    storage_level_series(sim_no_h2, REN_GW, "medium_storage_level"),
    label="Medium storage level",
    linewidth=1.2,
    color="#dd8452",
)
axes[1, 1].set_title("No H2: storage fluctuation")
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend(loc="upper right")

for ax in axes[1]:
    ax.set_xlabel("Winter date")
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

axes[0].plot(
    sim_h2.index,
    simulation_series(sim_h2, REN_GW, "gas_ccs_energy"),
    label="Gas with H2",
    linewidth=1.2,
    color="#c44e52",
)
axes[0].plot(
    sim_no_h2.index,
    simulation_series(sim_no_h2, REN_GW, "gas_ccs_energy"),
    label="Gas no H2",
    linewidth=1.2,
    color="#8172b3",
)
axes[0].plot(
    sim_h2.index,
    simulation_series(sim_h2, REN_GW, "dac_energy"),
    label="DAC with H2",
    linewidth=1.0,
    color="#64b5cd",
)
axes[0].plot(
    sim_no_h2.index,
    simulation_series(sim_no_h2, REN_GW, "dac_energy"),
    label="DAC no H2",
    linewidth=1.0,
    color="#4c72b0",
)
axes[0].set_title("Scenario reaction during the selected winter")
axes[0].set_ylabel("Energy per day (TWh)")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper right", ncol=2)

axes[1].plot(
    sim_h2.index,
    storage_level_series(sim_h2, REN_GW, "hydrogen_storage_level"),
    label="Hydrogen storage with H2",
    linewidth=1.2,
    color="#55a868",
)
axes[1].plot(
    sim_h2.index,
    storage_level_series(sim_h2, REN_GW, "medium_storage_level"),
    label="Medium storage with H2",
    linewidth=1.2,
    color="#dd8452",
)
axes[1].plot(
    sim_no_h2.index,
    storage_level_series(sim_no_h2, REN_GW, "medium_storage_level"),
    label="Medium storage no H2",
    linewidth=1.2,
    color="#937860",
)
axes[1].set_ylabel("Stored energy (TWh)")
axes[1].set_xlabel("Winter date")
axes[1].grid(alpha=0.3)
axes[1].legend(loc="upper right", ncol=2)
axes[1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

plt.tight_layout()
plt.show()
